In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from morph import *
from scipy import ndimage

mm.install()

#3B

img = cv2.imread('avatares.png')
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

s_channel = hsv[:,:,1]
v_channel = hsv[:,:,2]
_, mask_sat = cv2.threshold(s_channel, 25, 255, cv2.THRESH_BINARY)
gradient = cv2.morphologyEx(v_channel, cv2.MORPH_GRADIENT, np.ones((3,3), np.uint8))
_, mask_grad = cv2.threshold(gradient, 15, 255, cv2.THRESH_BINARY)
combined = cv2.bitwise_or(mask_sat, mask_grad)
mask_closed = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8), iterations=3)
mask_final = ndimage.binary_fill_holes(mask_closed).astype(np.uint8) * 255

dist_transform = cv2.distanceTransform(mask_final, cv2.DIST_L2, 5)
ret, sure_fg = cv2.threshold(dist_transform, 0.45 * dist_transform.max(), 255, 0)
sure_fg = np.uint8(sure_fg)
sure_bg = cv2.dilate(mask_final, np.ones((3,3), np.uint8), iterations=3)
unknown = cv2.subtract(sure_bg, sure_fg)
ret, markers = cv2.connectedComponents(sure_fg)
markers = markers + 1
markers[unknown == 255] = 0
markers_result = cv2.watershed(img, markers)

img_out = img.copy()

for label in np.unique(markers_result):
    if label <= 1:
        continue

    mask_obj = np.uint8(markers_result == label)

    contours, _ = cv2.findContours(mask_obj, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    cv2.drawContours(img_out, contours, -1, (0, 255, 0), 2)

    M = cv2.moments(contours[0])
    if M["m00"] != 0:
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])

        texto = f"Objeto {label-1}"

        cv2.putText(img_out, texto, (cX - 40, cY), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (0, 0, 0), 4, cv2.LINE_AA) # Borda preta
        cv2.putText(img_out, texto, (cX - 40, cY), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (255, 255, 255), 2, cv2.LINE_AA) # Texto branco

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(img_out, cv2.COLOR_BGR2RGB))
plt.title("3.b - Objetos Segmentados e Rotulados")
plt.axis('off')
plt.show()